# Run pywatershed with hydro-param output

End-to-end verification: load hydro-param's `parameters.nc`, forcing files,
and soltab into pywatershed v2.0 and run the full NHM process chain.

**Prerequisites:**
- Run this notebook in the `pws-test` pixi environment: `pixi run -e pws-test lab`
- Complete hydro-param pipeline + pywatershed derivation for a domain (e.g., DRB)

In [ ]:
from pathlib import Path

import numpy as np
import pywatershed as pws

## Configuration

Point `MODEL_DIR` at the pywatershed output from `hydro-param pywatershed run`.

In [ ]:
MODEL_DIR = Path("../pw-check/models/pywatershed")

# Simulation period (subset of forcing data for quick testing)
START = "1980-10-01"
END = "1981-09-30"  # 1 water year

print(f"Model dir: {MODEL_DIR}")
print(f"Contents: {sorted(p.name for p in MODEL_DIR.iterdir())}")

## Load parameters

In [ ]:
params = pws.parameters.PrmsParameters.from_netcdf(MODEL_DIR / "parameters.nc", use_xr=True)
print(f"Parameters: {len(params.data_vars)} variables")
print(f"Dimensions: {dict(params.dims)}")
print(f"\nVariable names:\n{sorted(params.data_vars.keys())}")

## Create control

In [ ]:
control = pws.Control(
    start_time=np.datetime64(f"{START}T00:00:00"),
    end_time=np.datetime64(f"{END}T00:00:00"),
    time_step=np.timedelta64(24, "h"),
    options={
        "input_dir": str(MODEL_DIR / "forcing"),
        "budget_type": "warn",
        "calc_method": "numpy",
    },
)
print(f"Control: {START} to {END}, daily timestep")

## Build and run model

In [ ]:
nhm_processes = [
    pws.PRMSSolarGeometry,
    pws.PRMSAtmosphere,
    pws.PRMSCanopy,
    pws.PRMSSnow,
    pws.PRMSRunoff,
    pws.PRMSSoilzone,
    pws.PRMSGroundwater,
    pws.PRMSChannel,
]

model = pws.Model(
    nhm_processes,
    control=control,
    parameters=params,
)
print("Model created successfully!")
print(f"Processes: {list(model.processes.keys())}")

In [ ]:
%%time
# Run all timesteps
try:
    model.run(finalize=True)
    n_steps = int((control.end_time - control.start_time) / control.time_step)
    print(f"Completed {n_steps} timesteps successfully!")
except Exception as e:
    print(f"Failed: {type(e).__name__}: {e}")
    raise

## Inspect results

Check key output variables from each process.

In [ ]:
# Summarize final state of each process
for proc_name, proc in model.processes.items():
    print(f"\n--- {proc_name} ---")
    # Show a few key output variables
    for var_name in list(proc.get_variables())[:5]:
        val = getattr(proc, var_name, None)
        if val is not None and hasattr(val, "shape"):
            finite = val[np.isfinite(val)]
            if len(finite) > 0:
                print(
                    f"  {var_name}: shape={val.shape}, "
                    f"range=[{finite.min():.4f}, {finite.max():.4f}], "
                    f"mean={finite.mean():.4f}"
                )

## Quick diagnostic plots

In [ ]:
# Channel flow at the outlet segment (last segment or max tosegment=0)
channel = model.processes["PRMSChannel"]
seg_outflow = getattr(channel, "seg_outflow", None)

if seg_outflow is not None:
    # Find outlet segment (tosegment == 0)
    toseg = params.data_vars["tosegment"]
    outlet_idx = np.where(toseg == 0)[0]
    if len(outlet_idx) > 0:
        print(f"Outlet segment index: {outlet_idx[0]}")
        print(f"Final outflow: {seg_outflow[outlet_idx[0]]:.2f} cfs")
    else:
        print("No outlet segment found (tosegment==0)")
else:
    print("seg_outflow not available")